In [11]:
import tensorflow as tf
import numpy as np

In [12]:
# Чтение текста
with open('input.txt', 'r', encoding='cp1251') as f:
    text = f.read()


print('Длина текста:', len(text))
print(text[:300])

Длина текста: 403725
Война и мир. Книга 1
Лев Николаевич Толстой


Война и мир #1
В книгу вошли первый и второй тома романа «Война и мир» – одного из самых знаменитых произведений литературы XIX века.






Том первый





Часть первая





I


– Eh bien, mon prince. G?nes et Lucques ne sont plus que des apanages, des п


In [13]:
# Уникальные символы
chars = sorted(list(set(text)))
vocab_size = len(chars)
print('Размер словаря:', vocab_size)


char2idx = {c: i for i, c in enumerate(chars)}
idx2char = np.array(chars)

Размер словаря: 146


In [14]:
# Преобразование текста в числа
text_as_int = np.array([char2idx[c] for c in text])

In [15]:
seq_length = 100
examples_per_epoch = len(text_as_int) // (seq_length + 1)


char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)


sequences = char_dataset.batch(seq_length + 1, drop_remainder=True)

In [16]:
# Разделение на вход и цель


def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text




dataset = sequences.map(split_input_target)

In [17]:
BATCH_SIZE = 64
BUFFER_SIZE = 10000


dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)

In [18]:
embedding_dim = 256
rnn_units = 1024

In [21]:
def build_model(vocab_size, embedding_dim, rnn_units, batch_size):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(batch_shape=(batch_size, None)),
        tf.keras.layers.Embedding(vocab_size, embedding_dim),
        tf.keras.layers.LSTM(rnn_units, return_sequences=True, stateful=True),
        tf.keras.layers.Dense(vocab_size)
    ])
    return model

In [22]:
model = build_model(vocab_size, embedding_dim, rnn_units, BATCH_SIZE)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (64, None, 256)        │        37,376 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (64, None, 1024)       │     5,246,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (64, None, 146)        │       149,650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,434,002 (20.73 MB)

 Trainable params: 5,434,002 (20.73 MB)

 Non-trainable params: 0 (0.00 B)

In [23]:
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
model.compile(optimizer='adam', loss=loss)

In [ ]:
EPOCHS = 20
history = model.fit(dataset, epochs=EPOCHS)

Epoch 1/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 63s 1s/step - loss: 3.4100
Epoch 2/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 38s 611ms/step - loss: 2.6429
Epoch 3/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 39s 618ms/step - loss: 2.4248
Epoch 4/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 39s 624ms/step - loss: 2.2761
Epoch 5/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 39s 618ms/step - loss: 2.1523
Epoch 6/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 38s 614ms/step - loss: 2.0450
Epoch 7/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 38s 616ms/step - loss: 1.9529
Epoch 8/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 38s 617ms/step - loss: 1.8730
Epoch 9/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 38s 616ms/step - loss: 1.8049
Epoch 10/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 38s 614ms/step - loss: 1.7469


In [25]:
model_gen = build_model(vocab_size, embedding_dim, rnn_units, batch_size=1)
model_gen.set_weights(model.get_weights())

In [ ]:
def generate_text(model, start_string, num_generate=1000, temperature=1.0):
    input_eval = [char2idx[s] for s in start_string]
    input_eval = tf.expand_dims(input_eval, 0)

    text_generated = []

    for _ in range(num_generate):
        predictions = model(input_eval)
        predictions = predictions[:, -1, :] / temperature
        predicted_id = tf.random.categorical(predictions, num_samples=1)[-1,0].numpy()
        input_eval = tf.expand_dims([predicted_id], 0)
        text_generated.append(idx2char[predicted_id])

    with open("generated_text.txt", "w", encoding="utf-8") as f:
        f.write(start_string + "".join(text_generated))
    return start_string + "".join(text_generated)


seed = "Приветствую"
print("Generated text:\n")
print(generate_text(model_gen, seed))

Generated text:

Приветствуюмами; Сонисоасной которы устояти на него гусарм.
Вый в авесно.

– Полком боснее счашать. Нашок бысть хорыш, и со малеть, новой друг сыно. – зпробаться поз видем, был ее; дам могу своех для мне Пьера.

– Полка и ондрачался, шашь.

– Насарась! Надъю, – скрывая с Пвернеги на голчая, малени седпервшаться он пещен стало похоеду.

– Нет, очень высопустил, чта прошать-езы непроят.

– Din! suvoid? les poutere? l’ive zout que ma n? pluce qu’av fascecil2 la paritinte del nnensiavateur; jous amagie qu’emre Etеne-ntruinn а mr?te, que ma carle lu nеpotbin,[21 -прожато. Всем сил на дурах. Он говырались замичал своей наждно голос ужехал, замеловая и извысканном жино столь, нопушка гударский своим мне опять отцец Las, la sitrе cCer jeu liis. Vous amant in s’y le suav il d’h?re des trinces, que qui c’harant touve, jakniatse aut resair les den souraice qu’illes dant un pasarellant, собщенщие нашого не мога, в дотю, и что-ны Родовая забратителья гестару, они ведна придольжал, 